# 7장. 과적합을 다시 보기: 이중 하강

모델을 키우면 새 자료의 오차가 커진다. 더 키우면 다시 작아진다. 그 사이에서 생기는 일을 확인한다.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 8분쯤 걸린다.

## 7.1 특성의 수를 늘리며 학습 오차와 2025년 오차 재기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
import torch
from torch import nn

def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

hour = (h.index - pd.Timedelta(hours=1)).hour             # 1시의 값이 0번 칸에 온다
hourly = h.assign(시=hour)
by_hour = hourly.pivot(index="날짜", columns="시", values="기온")
H23 = by_hour.loc[weekday.loc["2023"].index].to_numpy()        # (247, 24)
H25 = by_hour.loc[weekday.loc["2025"].index].to_numpy()        # (243, 24)

week = weekday.loc["2023"].index.isocalendar().week.to_numpy()
held = week % 4 == 1                           # 6장과 같은 방법으로 검증 자료를 뗀다
h_mean, h_std = H23[~held].mean(), H23[~held].std()   # 24칸을 한 값으로 표준화한다
X_tr, X_va, X25 = [(H - h_mean) / h_std for H in (H23[~held], H23[held], H25)]
Y_tr, Y_va = Y23[~held], Y23[held]
y_mean, y_std = Y_tr.mean(), Y_tr.std()
z_tr = (Y_tr - y_mean) / y_std

print(X_tr.shape, X_va.shape, X25.shape, X_tr.dtype)
print(np.round(H23[0], 1))
print(float(np.abs(H23.mean(axis=1) - T23).max()) < 1e-9, np.linalg.matrix_rank(X_tr))

In [ ]:
ones = lambda X: np.column_stack([X, np.ones(len(X))])       # 절편을 위한 1의 열
coef = np.linalg.lstsq(ones(X_tr), z_tr, rcond=None)[0]
for name, X, Y in [("학습", X_tr, Y_tr), ("검증", X_va, Y_va), ("2025년", X25, Y25)]:
    print(name, f"{rmse(Y, ones(X) @ coef * y_std + y_mean):.2f}")

In [ ]:
def features(seed, p):
    """무작위 가중치로 특성 p개를 만드는 함수를 돌려준다. 가중치는 고치지 않는다."""
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 1 / np.sqrt(24), (24, p))
    b = rng.normal(0, 1, p)
    return lambda X: np.maximum(0, X @ W + b)                 # ReLU

def gw(F, w):
    return F @ w * y_std + y_mean

def fit_exact(p, seed):
    """특성 p개의 가중치를 최소제곱법으로 구하고 결과를 사전으로 돌려준다."""
    phi = features(seed, p)
    F = phi(X_tr)                                             # (185, p)
    w, _, rank, _ = np.linalg.lstsq(F, z_tr, rcond=None)
    return dict(train=rmse(Y_tr, gw(F, w)), valid=rmse(Y_va, gw(phi(X_va), w)),
                test=rmse(Y25, gw(phi(X25), w)), norm=np.linalg.norm(w), rank=rank)

table = {}

def sweep(sizes):
    for p in sizes:
        runs = [fit_exact(p, seed) for seed in range(5)]
        table[p] = {key: np.array([run[key] for run in runs]) for key in runs[0]}
        t = table[p]
        span = f"({t['test'].min():,.2f}~{t['test'].max():,.2f})"
        print(f"{p:>6} {np.median(t['train']):>6.3f} {np.median(t['valid']):>10,.2f} "
              f"{np.median(t['test']):>10,.2f} {span:<22} "
              f"{np.median(t['norm']):>10,.1f} {int(np.median(t['rank'])):>4}")

F = features(0, 5)(X_tr)
print(F.shape, np.round(F[0], 2))
print("특성 수   학습       검증     2025년 (범위)                   가중치 크기  독립")
sweep([2, 5, 10, 20, 40, 80, 120, 150])

## 7.2 특성의 수가 자료의 수에 가까워질 때

In [ ]:
print("특성 수   학습       검증     2025년 (범위)                   가중치 크기  독립")
sweep([160, 170, 180, 185, 190, 200, 210, 220, 240, 260])

In [ ]:
F = features(0, 185)(X_tr)
off = int((F.max(axis=0) == 0).sum())              # 185일 모두에서 0인 특성
on = int((F.min(axis=0) > 0).sum())                # 185일 모두에서 0보다 큰 특성
print(off, on, 185 - off - on, np.linalg.matrix_rank(F))

In [ ]:
print("특성 수   독립    가장 큰 값    남긴 가장 작은 값               비     가중치 크기")
spectrum = {}
for p in [40, 150, 185, 220, 260, 500, 5000, 20000]:
    F = features(0, p)(X_tr)
    s = np.linalg.svd(F, compute_uv=False)         # 큰 것부터 min(185, p)개
    cut = np.finfo(float).eps * max(F.shape) * s[0]   # lstsq는 이보다 작으면 버린다
    kept = s[s > cut]
    w = np.linalg.lstsq(F, z_tr, rcond=None)[0]
    spectrum[p] = kept
    print(f"{p:>6} {len(kept):>6} {kept[0]:>12.1f} {kept[-1]:>20.2e} "
          f"{kept[0] / kept[-1]:>16.2e} {np.linalg.norm(w):>14,.1f}")

In [ ]:
F = features(0, 260)(X_tr)
U, s, Vt = np.linalg.svd(F, full_matrices=False)
match = U.T @ z_tr                             # 방향마다 정답과 맞는 정도
parts = match / s                              # 그 값을 특잇값으로 나눈다
w = Vt.T @ parts
print(np.allclose(w, np.linalg.lstsq(F, z_tr, rcond=None)[0]))
order = np.argsort(-np.abs(parts))[:3]
for k in order:
    print(f"{k + 1:>4}번째 방향: 특잇값 {s[k]:.2e}, 맞는 정도 {match[k]:+.4f}, "
          f"나눈 값 {parts[k]:+,.1f}")
print(f"{np.linalg.norm(parts[:100]):.2f} {np.linalg.norm(parts[100:]):.1f} "
      f"{np.linalg.norm(w):.1f}")
print(f"{np.abs(match[:100]).mean():.3f} {np.abs(match[100:]).mean():.3f}")

## 7.3 특성이 자료보다 많을 때 고르는 해

In [ ]:
print("특성 수   학습       검증     2025년 (범위)                   가중치 크기  독립")
sweep([350, 500, 1000, 2000, 5000, 20000])

In [ ]:
phi = features(0, 5000)
F = phi(X_tr)
w_min = np.linalg.lstsq(F, z_tr, rcond=None)[0]              # lstsq가 돌려주는 해
w_formula = F.T @ np.linalg.solve(F @ F.T, z_tr)             # 식으로 구한 가장 작은 해
print(f"{np.abs(w_min - w_formula).max():.1e}")

w_first = np.zeros(5000)                                     # 앞의 260개 특성만 쓰는 해
w_first[:260] = np.linalg.lstsq(F[:, :260], z_tr, rcond=None)[0]

for name, w in [("가장 작은 해", w_min), ("앞의 260개만 쓴 해", w_first)]:
    print(f"{name}: 학습 {rmse(Y_tr, gw(F, w)):.6f}, "
          f"2025년 {rmse(Y25, gw(phi(X25), w)):.2f}, 크기 {np.linalg.norm(w):.2f}")

## 7.4 규제와 경사하강법이 고르는 해

In [ ]:
def ridge(U, s, Vt, z, lam):
    """특잇값 분해로 릿지 회귀의 해를 구한다."""
    return Vt.T @ (s / (s ** 2 + lam) * (U.T @ z))

lams = [1e-8, 1e-6, 1e-4, 1e-2, 1.0, 100.0]
sizes = [40, 150, 185, 220, 260, 500, 5000]
ridged = {}
for p in sizes:
    for seed in range(5):
        phi = features(seed, p)
        U, s, Vt = np.linalg.svd(phi(X_tr), full_matrices=False)
        F_va, F25 = phi(X_va), phi(X25)
        for lam in lams:
            w = ridge(U, s, Vt, z_tr, lam)
            ridged.setdefault((lam, p), []).append(
                (rmse(Y_va, gw(F_va, w)), rmse(Y25, gw(F25, w))))
ridged = {key: np.array(rows) for key, rows in ridged.items()}

for title, col in [("검증 RMSE", 0), ("2025년 RMSE", 1)]:
    print(title + "  특성 수:" + "".join(f"{p:>9}" for p in sizes))
    for lam in lams:
        print(f"  λ = {lam:<8g}     " +
              "".join(f"{np.median(ridged[lam, p][:, col]):>9,.2f}" for p in sizes))
print("검증 RMSE가 가장 낮은 λ와 그 λ의 2025년 RMSE")
for p in sizes:
    lam = min(lams, key=lambda value: np.median(ridged[value, p][:, 0]))
    marks = ridged[lam, p][:, 1]
    span = f"({marks.min():.2f}~{marks.max():.2f})"
    print(f"{p:>6}  λ = {lam:<6g} {np.median(marks):.2f} {span}")

In [ ]:
phi = features(0, 2000)
F, F25 = phi(X_tr), phi(X25)
w_min = np.linalg.lstsq(F, z_tr, rcond=None)[0]
rate = 1 / np.linalg.norm(F, 2) ** 2              # 가장 큰 특잇값의 제곱의 역수
def to_rows(v):
    """벡터 v에서 F의 행을 더해서 만들 수 있는 부분만 남긴다."""
    return F.T @ np.linalg.solve(F @ F.T, F @ v)

w = np.zeros(2000)                                # 0에서 시작한다
marks = set(np.unique(np.logspace(0, 5, 120).astype(int)).tolist())
path = []
for step in range(1, 100001):
    w -= rate * F.T @ (F @ w - z_tr)              # 손실의 기울기는 F의 행을 더한 것이다
    if step in marks:
        path.append((step, rmse(Y_tr, gw(F, w)), rmse(Y25, gw(F25, w))))
    if step in (10, 100, 1000, 10000, 100000):
        apart = np.linalg.norm(w - w_min) / np.linalg.norm(w_min)
        outside = np.linalg.norm(w - to_rows(w)) / np.linalg.norm(w)
        print(f"{step:>7} {path[-1][1]:.3f} {path[-1][2]:.2f}  "
              f"크기 {np.linalg.norm(w):.3f}  가장 작은 해와의 거리 {apart:.3f}  "
              f"행 밖의 부분 {outside:.0e}")
path = np.array(path)
best_step = int(path[path[:, 2].argmin(), 0])
print(best_step, f"{path[:, 2].min():.2f}", f"{rmse(Y25, gw(F25, w_min)):.2f}",
      f"{np.linalg.norm(w_min):.3f}")

## 7.5 전체를 학습시키는 신경망에서 확인하기

In [ ]:
torch.set_num_threads(1)
as_tensor = lambda a: torch.tensor(a, dtype=torch.float32)
A_tr, A_va, A25 = as_tensor(X_tr), as_tensor(X_va), as_tensor(X25)
Z_tr = as_tensor(z_tr[:, None])
loss_fn = nn.MSELoss()

def train_net(width, steps, seed):
    """은닉층 1개짜리 신경망을 모두 학습시킨다. 전체 배치, Adam, 학습률 0.003."""
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(24, width), nn.ReLU(), nn.Linear(width, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=0.003)
    for _ in range(steps):
        optimizer.zero_grad()
        loss_fn(net(A_tr), Z_tr).backward()
        optimizer.step()
    with torch.no_grad():
        out = lambda A: net(A).numpy()[:, 0].astype(np.float64) * y_std + y_mean
        return rmse(Y_tr, out(A_tr)), rmse(Y_va, out(A_va)), rmse(Y25, out(A25))

nets = {}

def widths(sizes, steps):
    for width in sizes:
        rows = np.array([train_net(width, steps, seed) for seed in range(5)])
        nets[steps, width] = rows
        print(f"{width:>5} {26 * width + 1:>7,} {np.median(rows[:, 0]):>6.3f} "
              f"{np.median(rows[:, 1]):>6.2f} {np.median(rows[:, 2]):>6.2f} "
              f"({rows[:, 2].min():.2f}~{rows[:, 2].max():.2f})")

print("유닛  가중치 수   학습   검증  2025년 (범위)      3,000걸음")
widths([1, 2, 4, 7, 10, 20, 50, 100, 300, 1000, 3000], 3000)

In [ ]:
print("유닛  가중치 수   학습   검증  2025년 (범위)      20,000걸음")
widths([7, 50, 300, 1000], 20000)

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, torch.__version__)
print(platform.machine(), X_tr.dtype, torch.get_default_dtype())
print(torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 7-2
first = [2, 5, 10, 20, 40, 80, 120, 150]
fig, ax = plt.subplots(figsize=(7, 4.8))
for key, col, name, lw in [("train", NAVY, "학습 자료 185일", 1.8), ("valid", GRAY, "검증 자료 62일", 1.4), ("test", BRICK, "2025년 243일", 1.8)]:
    ax.plot(first, [np.median(table[p][key]) for p in first], color=col, lw=lw, marker="o", ms=4.5, label=name)
for p in first:
    mid = np.median(table[p]["test"])
    spot = dict(x=p * 0.92, y=mid, ha="right", va="center") if p == 120 else dict(x=p, y=mid + 0.22, ha="center", va="bottom")
    ax.text(s=f"{mid:.2f}", fontsize=9, color=BRICK, weight="bold", **spot)
ax.fill_between(first, [table[p]["test"].min() for p in first], [table[p]["test"].max() for p in first], color=BRICK, alpha=0.13, lw=0)
ax.set_xscale("log"); ax.set_xlim(1.8, 170); ax.set_ylim(0, 7.5); plain_ticks(ax.xaxis); ax.minorticks_off()
ax.set_xticks(first); ax.set_xticklabels([str(p) for p in first])
ax.set_xlabel("특성의 수 (로그 축)", fontsize=9); ax.set_ylabel("RMSE (GW). 점은 시드 5개의 중앙값, 띠는 2025년의 범위", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center", ncol=3)
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_2.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-3
every = sorted(table)
shown = [(40, GRAY, "특성 40개"), (150, TEAL, "특성 150개"), (220, BRICK, "특성 220개"), (500, NAVY, "특성 500개"), (20000, "#5b6470", "특성 20,000개")]
fig, axes = plt.subplots(2, 1, figsize=(7, 8.0))
for p, col, name in shown:
    axes[0].plot(np.arange(1, len(spectrum[p]) + 1), spectrum[p], color=col, lw=1.7, label=name, ls=(0, (4, 2)) if p == 20000 else "-")
axes[0].set_yscale("log"); axes[0].set_xlim(0, 190); axes[0].set_ylim(1e-9, 1e4)
axes[0].set_yticks([1e3, 1, 1e-3, 1e-6, 1e-9]); axes[0].set_yticklabels(["1,000", "1", "0.001", "0.000001", "0.000000001"]); axes[0].minorticks_off()
axes[0].set_xlabel("특잇값의 순서 (큰 것부터)", fontsize=9); axes[0].set_ylabel("특잇값 (로그 축)", fontsize=9)
axes[0].legend(frameon=False, fontsize=9, loc="lower left", ncol=2)
frame(axes[0], "① 특성 행렬의 특잇값")
axes[1].plot(every, [np.median(table[p]["norm"]) for p in every], color=BRICK, lw=1.8, marker="o", ms=4)
axes[1].axvline(185, color="#5b6470", lw=0.9, ls=(0, (2, 2)))
norms = [np.median(table[p]["norm"]) for p in every]
top = int(np.argmax(norms))
axes[1].text(every[top] * 1.25, norms[top], f"특성 {every[top]:,}개: {norms[top]:,.2f}", fontsize=9, color=BRICK, va="center", weight="bold")
axes[1].text(every[-1] * 0.95, norms[-1] * 0.55, f"특성 {every[-1]:,}개: {norms[-1]:,.2f}", fontsize=9, color=BRICK, ha="right", va="top", weight="bold")
axes[1].text(170, norms[0] * 1.2, "자료의 수 185", fontsize=9, color="#5b6470", ha="right", va="center")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlim(1.8, 24000)
plain_ticks(axes[1].xaxis); plain_ticks(axes[1].yaxis); axes[1].minorticks_off()
axes[1].set_xlabel("특성의 수 (로그 축)", fontsize=9); axes[1].set_ylabel("가중치의 크기 (로그 축, 시드 5개의 중앙값)", fontsize=9)
frame(axes[1], "② 학습한 가중치의 크기")
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_7_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-4
every = sorted(table)
fig, ax = plt.subplots(figsize=(7, 5.6))
ax.fill_between(every, [table[p]["test"].min() for p in every], [table[p]["test"].max() for p in every], color=BRICK, alpha=0.13, lw=0)
ax.plot(every, [np.median(table[p]["test"]) for p in every], color=BRICK, lw=1.8, marker="o", ms=4, label="2025년 RMSE (띠는 시드 5개의 범위)")
ax.plot(every, [max(np.median(table[p]["train"]), 0.05) for p in every], color=NAVY, lw=1.6, marker="o", ms=3.5, label="학습 RMSE (0.05보다 작은 값은 0.05에 그렸다)")
ax.axvline(185, color="#5b6470", lw=0.9, ls=(0, (2, 2)))
ax.text(170, 6e4, "자료의 수 185", fontsize=9, color="#5b6470", ha="right", va="center")
tests = [np.median(table[p]["test"]) for p in every]
top = int(np.argmax(tests)); low = int(np.argmin(tests))
ax.text(every[top] * 1.25, tests[top], f"특성 {every[top]:,}개: {tests[top]:,.2f}", fontsize=9, color=BRICK, va="center", weight="bold")
ax.text(every[low], tests[low] * 2.6, f"가장 낮은 곳\n특성 {every[low]:,}개: {tests[low]:.2f}", fontsize=9, color=BRICK, ha="center", va="bottom", weight="bold")
ax.text(every[-1] * 0.95, tests[-1] * 1.7, f"특성 {every[-1]:,}개: {tests[-1]:.2f}", fontsize=9, color=BRICK, ha="right", weight="bold")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(1.8, 24000); ax.set_ylim(0.04, 4e5)
plain_ticks(ax.xaxis); plain_ticks(ax.yaxis); ax.minorticks_off()
ax.set_xlabel("특성의 수 (로그 축)", fontsize=9); ax.set_ylabel("RMSE (GW, 로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_4.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-6
fig, axes = plt.subplots(3, 1, figsize=(7, 9.2))
for ax, p, col in zip(axes, [40, 260, 20000], [TEAL, BRICK, NAVY]):
    phi = features(0, p)
    w = np.linalg.lstsq(phi(X_tr), z_tr, rcond=None)[0]
    guess = gw(phi(X25), w)
    ax.plot([50, 100], [50, 100], color="#5b6470", lw=0.9, ls=(0, (2, 2)))
    ax.scatter(Y25, np.clip(guess, 41, 109), s=10, color=col, alpha=0.75, lw=0)
    out = int(((guess < 40) | (guess > 110)).sum())
    ax.text(0.02, 0.93, f"특성 {p:,}개: 2025년 RMSE {rmse(Y25, guess):.2f}GW" + (f"\n그림 밖의 날 {out}일은 위아래 끝에 그렸다" if out else ""),
            transform=ax.transAxes, fontsize=9, color=col, va="top", weight="bold")
    ax.set_xlim(50, 85); ax.set_ylim(40, 110)
    ax.set_xlabel("2025년의 실제 전력수요 (GW)", fontsize=9); ax.set_ylabel("예측 (GW)", fontsize=9)
    frame(ax)
fig.tight_layout(h_pad=1.6)
fig.savefig("fig_7_6.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-7
looks = [(1e-8, GRAY, "λ = 0.00000001"), (1e-4, TEAL, "λ = 0.0001"), (1e-2, NAVY, "λ = 0.01"), (1.0, "#5b6470", "λ = 1")]
fig, ax = plt.subplots(figsize=(7, 5.0))
ax.plot(sizes, [np.median(table[p]["test"]) for p in sizes], color=BRICK, lw=1.8, marker="o", ms=4.5,
        label=f"규제 없음: 특성 {sizes[-1]:,}개에서 {np.median(table[sizes[-1]]['test']):.2f}")
for lam, col, name in looks:
    ax.plot(sizes, [np.median(ridged[lam, p][:, 1]) for p in sizes], color=col, lw=1.6, marker="s", ms=4,
            label=f"{name}: 특성 {sizes[-1]:,}개에서 {np.median(ridged[lam, sizes[-1]][:, 1]):.2f}", ls=(0, (4, 2)) if lam == 1.0 else "-")
ax.axvline(185, color="#5b6470", lw=0.9, ls=(0, (2, 2)))
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(34, 6000); ax.set_ylim(2, 2e4)
plain_ticks(ax.xaxis); plain_ticks(ax.yaxis); ax.minorticks_off()
ax.set_xticks([40, 150, 220, 500, 5000]); ax.set_xticklabels(["40", "150", "220", "500", "5,000"])
ax.text(178, 1.2e4, "자료의 수 185", fontsize=9, color="#5b6470", ha="right", va="center")
ax.set_xlabel("특성의 수 (로그 축)", fontsize=9); ax.set_ylabel("2025년 RMSE (GW, 로그 축, 시드 5개의 중앙값)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-8
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.plot(path[:, 0], path[:, 1], color=NAVY, lw=1.8, label="학습 RMSE")
ax.plot(path[:, 0], path[:, 2], color=BRICK, lw=1.8, label="2025년 RMSE")
ax.axhline(rmse(Y25, gw(F25, w_min)), color=BRICK, lw=0.9, ls=(0, (2, 2)))
ax.text(130, rmse(Y25, gw(F25, w_min)) + 0.08, f"가장 작은 해의 2025년 RMSE {rmse(Y25, gw(F25, w_min)):.2f}", fontsize=9, color=BRICK)
ax.scatter([best_step], [path[:, 2].min()], s=44, color=BRICK, zorder=5, edgecolor="white", lw=0.8)
ax.annotate(f"가장 낮은 곳\n({best_step:,}걸음, {path[:, 2].min():.2f})", (best_step, path[:, 2].min()), (best_step * 0.5, 3.05), fontsize=9, color=BRICK, ha="center",
            arrowprops=dict(arrowstyle="-", color=BRICK, lw=0.7))
ax.set_xscale("log"); ax.set_xlim(1, 100000); ax.set_ylim(0, 6.5); plain_ticks(ax.xaxis); ax.minorticks_off()
ax.set_xlabel("경사하강법의 걸음 (로그 축, 특성 2,000개, 시드 0)", fontsize=9); ax.set_ylabel("RMSE (GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 7-9
fig, ax = plt.subplots(figsize=(7, 5.0))
for steps, col, marker, name in [(3000, TEAL, "o", "3,000걸음"), (20000, BRICK, "s", "20,000걸음")]:
    sizes_here = sorted(width for (count, width) in nets if count == steps)
    xs = [26 * width + 1 for width in sizes_here]
    ax.fill_between(xs, [nets[steps, width][:, 2].min() for width in sizes_here], [nets[steps, width][:, 2].max() for width in sizes_here], color=col, alpha=0.13, lw=0)
    ax.plot(xs, [np.median(nets[steps, width][:, 2]) for width in sizes_here], color=col, lw=1.8, marker=marker, ms=4.5, label=name)
    mids = [np.median(nets[steps, width][:, 2]) for width in sizes_here]
    for k in {0, int(np.argmax(mids)), int(np.argmin(mids)), len(mids) - 1} - {k for k, mid in enumerate(mids) if mid > 4.8}:
        ax.text(xs[k], mids[k] + (0.09 if steps == 20000 else -0.09), f"{mids[k]:.2f}", fontsize=9, color=col, ha="center",
                va="bottom" if steps == 20000 else "top", weight="bold")
ax.axvline(185, color="#5b6470", lw=0.9, ls=(0, (2, 2)))
ax.text(200, 4.9, "자료의 수 185", fontsize=9, color="#5b6470", va="top")
ax.set_xscale("log"); ax.set_xlim(20, 110000); ax.set_ylim(2, 5); plain_ticks(ax.xaxis); ax.minorticks_off()
ax.set_xlabel("가중치와 편향의 수 (로그 축)", fontsize=9); ax.set_ylabel("2025년 RMSE (GW). 점은 시드 5개의 중앙값, 띠는 범위", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_9.png", dpi=300)
plt.show()